In [1]:
import polars as pl
import os
import glob

## Functions

In [ ]:
def reorder_cols(df: pl.DataFrame) -> pl.DataFrame:
	cols = df.columns

	if "sample" in cols:
		cols.remove("sample")
		cols.insert(0, "sample")

	return df[cols]

In [ ]:
def summarize_res(df: pl.DataFrame) -> pl.DataFrame:
	
	bool_cols = df['filter_1_mutation_intra_hairpin_loop':'filter_8_low_quality'].columns
	str_cols = df['msec_filter_123':'msec_filter_all'].columns
	n_variants = df.height

	# 1. Initialize a dictionary with empty lists for your columns
	results = {
		"filter": [],
		"percentage": [],
		"n_failed": []
	}

	# 2. Populate the dictionary inside your loops
	for col in bool_cols:
		n_failed = df[col].sum() # Sum counts True values
		
		results["filter"].append(f"%_{col}")
		results["percentage"].append((n_failed / n_variants) * 100)
		results["n_failed"].append(n_failed)

	for col in str_cols:
		n_failed = df.filter(~pl.col(col).is_null()).height # Count non-nulls
		
		results["filter"].append(f"%_{col}")
		results["percentage"].append((n_failed / n_variants) * 100)
		results["n_failed"].append(n_failed)

	# 3. Create DataFrame directly from the dictionary
	return pl.DataFrame(results).with_columns(pl.lit(n_variants).alias("total_varints"))

### MicroSEC filter Description

The MicroSEC pipeline contains 8 filtering processes.  

- Filter 1  : Shorter-supporting lengths distribute too short to occur (1-1 and 1-2).  
	- Filter 1-1: P-values are less than the threshold_p(default: 10^(-6)).  
	- Filter 1-2: The shorter-supporting lengths distributed over less than 75% of the read length.  
- Filter 2  : Hairpin-structure induced error detection (2-1 and 2-2).  
	- Filter 2-1: Palindromic sequences exist within 200 bases.  
	- Filter 2-2: >=50% mutation-supporting reads contains a reverse complementary sequence of the opposite strand consisting >= 15 bases.  
- Filter 3  : 3'-/5'-supporting lengths are too densely distributed to occur (3-1 and 3-2).  
	- Filter 3-1: P-values are less than the threshold_p(default: 10^(-6)).  
	- Filter 3-2: The distributions of 3'-/5'-supporting lengths are within 75% of the read length.  
- Filter 4  : >=15% mutations were called by chimeric reads comprising two distant regions.  
- Filter 5  : >=50% mutations were called by soft-clipped reads.  
- Filter 6  : Mutations locating at simple repeat sequences.  
- Filter 7  : Indel mutations locating at a >=15 homopolymer.  
- Filter 8  : >=10% of bases are low quality (Quality score <18) in the mutation supporting reads.  

Filter 1, 2, 3, and 4 detect possible FFPE artifacts.  
Filter 5 may also be FFPE artifacts or mapping errors.  
Filter 6, 7, and 8 detect frequent errors caused by the next generation sequencing platform.  
Supporting lengths are adjusted considering small repeat sequences around the mutations.  
  
Results are saved in a tsv file.  

github url: https://github.com/MANO-B/MicroSEC

## Main
### VCF

In [ ]:
msec_paths = sorted(glob.glob("../vcf-micr-svf/*/*.microsec.tsv"))

all_res = []

for i, path in enumerate(msec_paths):
	sample = os.path.basename(path).replace(".microsec.tsv", "")
	df = pl.read_csv(path, separator="\t").rename(lambda x : x.lower())
	
	all_res.append(df)
	print(f"{i+1} Processed {sample}")
	
final_df = pl.concat(all_res, how="vertical_relaxed").pipe(reorder_cols)

1 Processed ORD-1887498-01
2 Processed ORD-1888382-01
3 Processed ORD-1888385-01
4 Processed ORD-1888387-01
5 Processed ORD-1888390-01
6 Processed ORD-1889422-01
7 Processed ORD-1889460-01
8 Processed ORD-1889503-01
9 Processed ORD-1889504-01
10 Processed ORD-1889505-01
11 Processed ORD-1889526-01
12 Processed ORD-1892831-01
13 Processed ORD-1892833-01
14 Processed ORD-1893842-01
15 Processed ORD-1893882-01
16 Processed ORD-1893888-01
17 Processed ORD-1893890-01
18 Processed ORD-1893989-01
19 Processed ORD-1893994-01
20 Processed ORD-1894015-01
21 Processed ORD-1894057-01
22 Processed ORD-1894354-01
23 Processed ORD-1894818-01
24 Processed ORD-1894933-01
25 Processed ORD-1895615-01
26 Processed ORD-1895904-01
27 Processed ORD-1896836-01
28 Processed ORD-1896838-01
29 Processed ORD-1896953-01
30 Processed ORD-1896959-01
31 Processed ORD-1896970-01
32 Processed ORD-1897970-01
33 Processed ORD-1898635-01
34 Processed ORD-1899276-01
35 Processed ORD-1899371-01
36 Processed ORD-1899997-01
3

In [5]:
# All artifacts
arti_all_filter = final_df.filter(~pl.col("msec_filter_all").is_null())
display(arti_all_filter)
# Number of Samples with >= 1 artifacts
arti_all_filter["sample"].n_unique()

sample,mut_type,chr,pos,ref,alt,simplerepeat_trf,neighborhood_sequence,read_length,total_read,soft_clipped_read,flag_hairpin,pre_support_length,post_support_length,short_support_length,pre_farthest,post_farthest,low_quality_base_rate_under_q18,low_quality_pre,low_quality_post,distant_homology_rate,soft_clipped_rate,prob_filter_1,prob_filter_3_pre,prob_filter_3_post,filter_1_mutation_intra_hairpin_loop,filter_2_hairpin_structure,filter_3_microhomology_induced_mutation,filter_4_highly_homologous_region,filter_5_soft_clipped_reads,filter_6_simple_repeat,filter_7_mutation_at_homopolymer,filter_8_low_quality,msec_filter_123,msec_filter_1234,msec_filter_all,comment
str,str,str,i64,str,str,str,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,bool,bool,bool,bool,bool,bool,bool,bool,str,str,str,str
"""ORD-1887498-01""","""4-ins""","""chr10""",8115668,"""T""","""TAAAA""","""N""","""TTTCAGAGGCAGCAAAAAAGTAAAAAAAAA…",101,7,0,0,65,78,47,65,180,0.053748,0.014286,0.0,0.0,0.0,0.009092,0.011092,0.009525,false,false,false,false,false,false,true,false,null,null,"""Artifact suspicious""",null
"""ORD-1887498-01""","""2-del""","""chr17""",41218829,"""CTT""","""C""","""N""","""CAGCTCACCACCCTCCAAACCTTTTTTTTT…",101,36,4,0,77,99,50,185,163,0.181243,0.080556,0.316667,0.0,0.111111,0.071144,0.27186,0.174066,false,false,false,false,false,false,true,true,null,null,"""Artifact suspicious""",null
"""ORD-1887498-01""","""3-ins""","""chr3""",37067099,"""A""","""ATTT""","""Y""","""ATATATATATATATATATATATTTTTTTTT…",101,5,5,0,21,75,21,52,155,0.314851,0.12,0.22,0.4,1.0,0.104358,0.005916,0.012442,false,false,false,true,true,true,true,true,null,"""Artifact suspicious""","""Artifact suspicious""",null
"""ORD-1887498-01""","""4-ins""","""chr3""",187439746,"""T""","""TTATA""","""Y""","""TAGGTTTATATATATTTATTTTATATATAT…",101,348,4,0,76,92,49,335,286,0.033316,0.031897,0.042529,0.0,0.011494,1.0,0.000005,0.004258,false,false,false,false,false,true,false,false,null,null,"""Artifact suspicious""",null
"""ORD-1887498-01""","""6-ins""","""chr3""",187439746,"""T""","""TTATATA""","""Y""","""TAGGTTTATATATATTTATTTTATATATAT…",101,13,0,0,64,80,47,153,163,0.027418,0.015385,0.023077,0.0,0.0,0.067995,0.006784,0.010085,false,false,false,false,false,true,false,false,null,null,"""Artifact suspicious""",null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ORD-2131606-01""","""1-snv""","""chr17""",29483000,"""G""","""T""","""N""","""TTTTTTTTCTTTTTTTTTCATCTTCCAATA…",144,2299,464,0,143,143,71,599,596,0.070046,0.115789,0.038669,0.0,0.201827,1.0,1.0,1.0,false,false,false,false,false,false,false,true,null,null,"""Artifact suspicious""",null
"""ORD-2131606-01""","""16-del""","""chr2""",225422558,"""TTTCATCCATGGTCATC""","""T""","""N""","""CCAAATGCTGTTTACATATTTTGTAATATC…",144,185,52,0,125,126,80,277,141,0.069294,0.115676,0.046486,0.0,0.281081,2.0150e-34,6.6014e-10,0.000001,true,false,false,false,false,false,false,true,"""Artifact suspicious""","""Artifact suspicious""","""Artifact suspicious""",""" filter 3: p is small, but sup…"
"""ORD-2131606-01""","""2-ins""","""chr8""",117864264,"""C""","""CAG""","""N""","""TGGTGGAGGCATAGCTGACTCAGATCTATG…",144,37,26,0,118,112,56,145,112,0.010323,0.005405,0.010811,0.0,0.702703,9.2195e-9,0.000591,0.001694,true,false,false,false,true,false,false,false,"""Artifact suspicious""","""Artifact suspicious""","""Artifact suspicious""",null


288

In [6]:
# Filter 1234
arti_filter_1234 = final_df.filter(~pl.col("msec_filter_1234").is_null())
display(arti_filter_1234)
# Number of Samples with >= 1 artifacts
arti_filter_1234["sample"].n_unique()

sample,mut_type,chr,pos,ref,alt,simplerepeat_trf,neighborhood_sequence,read_length,total_read,soft_clipped_read,flag_hairpin,pre_support_length,post_support_length,short_support_length,pre_farthest,post_farthest,low_quality_base_rate_under_q18,low_quality_pre,low_quality_post,distant_homology_rate,soft_clipped_rate,prob_filter_1,prob_filter_3_pre,prob_filter_3_post,filter_1_mutation_intra_hairpin_loop,filter_2_hairpin_structure,filter_3_microhomology_induced_mutation,filter_4_highly_homologous_region,filter_5_soft_clipped_reads,filter_6_simple_repeat,filter_7_mutation_at_homopolymer,filter_8_low_quality,msec_filter_123,msec_filter_1234,msec_filter_all,comment
str,str,str,i64,str,str,str,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,bool,bool,bool,bool,bool,bool,bool,bool,str,str,str,str
"""ORD-1887498-01""","""3-ins""","""chr3""",37067099,"""A""","""ATTT""","""Y""","""ATATATATATATATATATATATTTTTTTTT…",101,5,5,0,21,75,21,52,155,0.314851,0.12,0.22,0.4,1.0,0.104358,0.005916,0.012442,false,false,false,true,true,true,true,true,null,"""Artifact suspicious""","""Artifact suspicious""",null
"""ORD-1888387-01""","""9-del""","""chr17""",7578545,"""CAGGGGAGTA""","""C""","""N""","""GCAAAACATCTTGTTGAGGGCCTGTAGGAA…",144,62,4,0,128,110,60,128,110,0.023858,0.030645,0.019355,0.0,0.064516,1.9162e-12,3.8711e-7,1.6516e-7,true,false,false,false,false,false,false,false,"""Artifact suspicious""","""Artifact suspicious""","""Artifact suspicious""",""" filter 3: p is small, but sup…"
"""ORD-1889526-01""","""1-snv""","""chr22""",29090061,"""G""","""A""","""N""","""GGCTTCTTCTGTCGTAAAACATGCCTTTGG…",144,67,12,0,140,143,66,274,161,0.035448,0.038806,0.038806,0.58209,0.179104,0.017276,0.352409,0.251762,false,false,false,true,false,false,false,false,null,"""Artifact suspicious""","""Artifact suspicious""",null
"""ORD-1894057-01""","""1-ins""","""chr4""",106196428,"""T""","""TA""","""N""","""AACTCTTCACACACTTCAGATAATCTATGG…",144,27,0,0,108,112,67,142,144,0.035751,0.033333,0.040741,0.0,0.0,1.5402e-10,3.0114e-10,2.1379e-9,true,false,true,false,false,false,false,false,"""Artifact suspicious""","""Artifact suspicious""","""Artifact suspicious""",null
"""ORD-1894354-01""","""2-del""","""chr1""",156845492,"""AGT""","""A""","""Y""","""CTGGGTCAAGGGCAGGGACGAGTGTGTGTG…",101,532,14,0,63,100,52,472,491,0.044778,0.020865,0.078383,0.996241,0.026316,1.0,3.1299e-8,1.0649e-39,false,false,false,true,false,true,false,false,null,"""Artifact suspicious""","""Artifact suspicious""",""" filter 3: p is small, but sup…"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ORD-2117808-01""","""1-snv""","""chr3""",178952085,"""A""","""G""","""N""","""GAAACAAATGAATGATGCACGTCATGGTGG…",101,29,0,0,60,90,49,219,326,0.017071,0.013793,0.027586,0.0,0.0,0.001003,3.1723e-9,3.0091e-9,false,false,true,false,false,false,false,false,"""Artifact suspicious""","""Artifact suspicious""","""Artifact suspicious""",null
"""ORD-2118842-01""","""1-snv""","""chr2""",25463568,"""A""","""G""","""N""","""CATTGCAGGGACTGCCCCCAGTCACCAGAT…",144,56,2,0,95,117,71,156,117,0.008433,0.010714,0.0125,0.0,0.035714,3.0944e-7,4.7255e-11,9.2555e-11,false,false,true,false,false,false,false,false,"""Artifact suspicious""","""Artifact suspicious""","""Artifact suspicious""",""" filter 1: p is small, but sup…"
"""ORD-2131606-01""","""16-del""","""chr2""",225422558,"""TTTCATCCATGGTCATC""","""T""","""N""","""CCAAATGCTGTTTACATATTTTGTAATATC…",144,185,52,0,125,126,80,277,141,0.069294,0.115676,0.046486,0.0,0.281081,2.0150e-34,6.6014e-10,0.000001,true,false,false,false,false,false,false,true,"""Artifact suspicious""","""Artifact suspicious""","""Artifact suspicious""",""" filter 3: p is small, but sup…"


88

In [7]:
summarize_res(final_df)

filter,percentage,n_failed,total_varints
str,f64,i64,i32
"""%_filter_1_mutation_intra_hair…",0.594335,47,7908
"""%_filter_2_hairpin_structure""",0.0,0,7908
"""%_filter_3_microhomology_induc…",0.657562,52,7908
"""%_filter_4_highly_homologous_r…",0.379363,30,7908
"""%_filter_5_soft_clipped_reads""",0.847243,67,7908
…,…,…,…
"""%_filter_7_mutation_at_homopol…",0.189681,15,7908
"""%_filter_8_low_quality""",2.364694,187,7908
"""%_msec_filter_123""",1.024279,81,7908


### XML

In [22]:
msec_paths = sorted(glob.glob("../xml-micr-svf/*/*.microsec.tsv"))
len(msec_paths)

535

In [23]:
all_res = []

for i, path in enumerate(msec_paths):
	sample = os.path.basename(path).replace(".microsec.tsv", "")
	df = pl.read_csv(path, separator="\t").rename(lambda x: x.lower())
	
	all_res.append(df)
	print(f"{i+1} Processed {sample}")
	
final_df = pl.concat(all_res, how="vertical_relaxed").pipe(reorder_cols)

1 Processed ORD-1887498-01
2 Processed ORD-1888382-01
3 Processed ORD-1888385-01
4 Processed ORD-1888387-01
5 Processed ORD-1888390-01
6 Processed ORD-1889422-01
7 Processed ORD-1889460-01
8 Processed ORD-1889503-01
9 Processed ORD-1889504-01
10 Processed ORD-1889505-01
11 Processed ORD-1889526-01
12 Processed ORD-1892831-01
13 Processed ORD-1892833-01
14 Processed ORD-1893842-01
15 Processed ORD-1893882-01
16 Processed ORD-1893888-01
17 Processed ORD-1893890-01
18 Processed ORD-1893989-01
19 Processed ORD-1893994-01
20 Processed ORD-1894015-01
21 Processed ORD-1894057-01
22 Processed ORD-1894354-01
23 Processed ORD-1894818-01
24 Processed ORD-1894933-01
25 Processed ORD-1895615-01
26 Processed ORD-1895904-01
27 Processed ORD-1896836-01
28 Processed ORD-1896838-01
29 Processed ORD-1896953-01
30 Processed ORD-1896959-01
31 Processed ORD-1896970-01
32 Processed ORD-1897970-01
33 Processed ORD-1898635-01
34 Processed ORD-1899276-01
35 Processed ORD-1899371-01
36 Processed ORD-1899997-01
3

In [24]:
final_df

sample,chrom,pos,ref,alt,gene,is_vus,depth,cds_effect,protein_effect,allele_fraction,functional_effect,transcript,strand,equivocal,mut_type,simplerepeat_trf,neighborhood_sequence,read_length,total_read,soft_clipped_read,flag_hairpin,pre_support_length,post_support_length,short_support_length,pre_farthest,post_farthest,low_quality_base_rate_under_q18,low_quality_pre,low_quality_post,distant_homology_rate,soft_clipped_rate,prob_filter_1,prob_filter_3_pre,prob_filter_3_post,filter_1_mutation_intra_hairpin_loop,filter_2_hairpin_structure,filter_3_microhomology_induced_mutation,filter_4_highly_homologous_region,filter_5_soft_clipped_reads,filter_6_simple_repeat,filter_7_mutation_at_homopolymer,filter_8_low_quality,msec_filter_123,msec_filter_1234,msec_filter_all,comment
str,str,i64,str,str,str,bool,i64,str,str,f64,str,str,str,bool,str,str,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,bool,bool,bool,bool,bool,bool,bool,bool,str,str,str,str
"""ORD-1887498-01""","""chr12""",69222553,"""G""","""C""","""MDM2""",true,2653,"""526G>C""","""E176Q""",0.0682,"""missense""","""NM_002392""","""+""",false,"""1-snv""","""N""","""TTTTCCTTACATATCCAGAACAAAATTCAG…",101,498,12,0,100,100,50,195,398,0.033421,0.045382,0.0251,0.0,0.024096,1.0,1.0,1.0,false,false,false,false,false,false,false,false,null,null,null,null
"""ORD-1887498-01""","""chr21""",39775625,"""G""","""A""","""ERG""",true,1258,"""395C>T""","""T132M""",0.1892,"""missense""","""NM_182918""","""-""",false,"""1-snv""","""N""","""CATGGTCTGTACTCCATAGCATAGGATCTG…",101,571,15,0,100,100,50,365,280,0.030067,0.030823,0.036953,0.0,0.02627,1.0,1.0,1.0,false,false,false,false,false,false,false,false,null,null,null,null
"""ORD-1887498-01""","""chr4""",1803138,"""C""","""G""","""FGFR3""",true,886,"""490C>G""","""L164V""",0.3679,"""missense""","""NM_000142""","""+""",false,"""1-snv""","""N""","""AGCGGATGGACAAGAAGCTGGTGGCCGTGC…",101,745,11,0,100,100,50,238,389,0.025078,0.024161,0.032483,0.0,0.014765,1.0,1.0,1.0,false,false,false,false,false,false,false,false,null,null,null,null
"""ORD-1887498-01""","""chr7""",55259515,"""T""","""G""","""EGFR""",false,1529,"""2573T>G""","""L858R""",0.4042,"""missense""","""NM_005228""","""+""",false,"""1-snv""","""N""","""CAAGATCACAGATTTTGGGCGGGCCAAACT…",101,1468,20,0,100,101,50,409,322,0.030033,0.031403,0.029632,0.0,0.013624,1.0,1.0,1.0,false,false,false,false,false,false,false,false,null,null,null,null
"""ORD-1888382-01""","""chr12""",49437724,"""T""","""A""","""MLL2""",true,1004,"""5246A>T""","""D1749V""",0.004,"""missense""","""NM_003482""","""-""",false,"""1-snv""","""N""","""GCTGTTGCTTCTTCTTCTCAACCCCTTCAG…",144,71,21,0,143,135,71,294,272,0.104069,0.122535,0.061972,0.0,0.295775,1.0,0.08454,0.321938,false,false,false,false,false,false,false,true,null,null,"""Artifact suspicious""",null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ORD-2133291-01""","""chr17""",37856504,"""G""","""A""","""ERBB2""",true,3682,"""13G>A""","""A5T""",0.0019,"""missense""","""NM_004448""","""+""",false,"""1-snv""","""N""","""TGAGCACCATGGAGCTGGCGACCTTGTGCC…",144,195,1,0,133,139,69,179,414,0.039672,0.048718,0.010256,0.0,0.005128,1.2908e-7,1.7741e-8,1.2642e-8,false,false,false,false,false,false,false,false,null,null,null,""" filter 1: p is small, but sup…"
"""ORD-2133291-01""","""chr22""",29108003,"""C""","""T""","""CHEK2""",true,3729,"""686G>A""","""G229D""",0.0013,"""missense""","""NM_007194""","""-""",false,"""1-snv""","""N""","""GCTTTACCTCTCCACAGGCATCACTAGAGG…",144,36,0,0,140,140,52,140,338,0.009452,0.016667,0.005556,0.0,0.0,0.000018,0.405816,0.401138,false,false,false,false,false,false,false,false,null,null,null,null
"""ORD-2133291-01""","""chr4""",1920021,"""A""","""C""","""WHSC1""",true,1979,"""1081A>C""","""K361Q""",0.5073,"""missense""","""NM_133335""","""+""",false,"""1-snv""","""N""","""ATCTCAACCCTCAAGTAGCCCAGGAGGCTG…",144,7806,748,0,143,143,71,703,734,0.013053,0.014758,0.015168,0.0,0.095824,1.0,1.0,1.0,false,

In [25]:
# All artifacts
arti_all_filter = final_df.filter(~pl.col("msec_filter_all").is_null())
display(arti_all_filter)
# Number of Samples with >= 1 artifacts
arti_all_filter["sample"].n_unique()

sample,chrom,pos,ref,alt,gene,is_vus,depth,cds_effect,protein_effect,allele_fraction,functional_effect,transcript,strand,equivocal,mut_type,simplerepeat_trf,neighborhood_sequence,read_length,total_read,soft_clipped_read,flag_hairpin,pre_support_length,post_support_length,short_support_length,pre_farthest,post_farthest,low_quality_base_rate_under_q18,low_quality_pre,low_quality_post,distant_homology_rate,soft_clipped_rate,prob_filter_1,prob_filter_3_pre,prob_filter_3_post,filter_1_mutation_intra_hairpin_loop,filter_2_hairpin_structure,filter_3_microhomology_induced_mutation,filter_4_highly_homologous_region,filter_5_soft_clipped_reads,filter_6_simple_repeat,filter_7_mutation_at_homopolymer,filter_8_low_quality,msec_filter_123,msec_filter_1234,msec_filter_all,comment
str,str,i64,str,str,str,bool,i64,str,str,f64,str,str,str,bool,str,str,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,bool,bool,bool,bool,bool,bool,bool,bool,str,str,str,str
"""ORD-1888382-01""","""chr12""",49437724,"""T""","""A""","""MLL2""",true,1004,"""5246A>T""","""D1749V""",0.004,"""missense""","""NM_003482""","""-""",false,"""1-snv""","""N""","""GCTGTTGCTTCTTCTTCTCAACCCCTTCAG…",144,71,21,0,143,135,71,294,272,0.104069,0.122535,0.061972,0.0,0.295775,1.0,0.08454,0.321938,false,false,false,false,false,false,false,true,null,null,"""Artifact suspicious""",null
"""ORD-1888382-01""","""chr9""",139440221,"""CG""","""AA""","""NOTCH1""",true,434,"""17_18CG>TT""","""A6V""",0.5876,"""missense""","""NM_017617""","""-""",false,"""2-snv""","""N""","""AGCGCCAGGCAGAGCAGGGGAACCAGGAGC…",144,2125,546,0,138,138,71,474,215,0.069941,0.018824,0.157976,0.0,0.256941,0.000018,0.000018,1.0,false,false,false,false,false,false,false,true,null,null,"""Artifact suspicious""",null
"""ORD-1888387-01""","""chr17""",7578545,"""CAGGGGAGTA""","""C""","""TP53""",false,6789,"""376_384delTACTCCCCT""","""Y126_P128del""",0.001,"""nonframeshift""","""NM_000546""","""-""",false,"""9-del""","""N""","""GCAAAACATCTTGTTGAGGGCCTGTAGGAA…",144,62,4,0,128,110,60,128,110,0.023858,0.030645,0.019355,0.0,0.064516,1.9162e-12,3.8711e-7,1.6516e-7,true,false,false,false,false,false,false,false,"""Artifact suspicious""","""Artifact suspicious""","""Artifact suspicious""",""" filter 3: p is small, but sup…"
"""ORD-1888390-01""","""chr16""",3779278,"""C""","""T""","""CREBBP""",true,926,"""5770G>A""","""V1924M""",0.5,"""missense""","""NM_004380""","""-""",false,"""1-snv""","""N""","""GGGGGGCTGAGTCCGGGCCATGCTGGGGAA…",144,4573,1469,0,143,143,71,460,455,0.105032,0.019484,0.176427,0.0,0.321233,1.0,1.0,1.0,false,false,false,false,false,false,false,true,null,null,"""Artifact suspicious""",null
"""ORD-1888390-01""","""chr19""",1219381,"""G""","""C""","""STK11""",true,5353,"""433G>C""","""E145Q""",0.0017,"""missense""","""NM_000455""","""+""",false,"""1-snv""","""N""","""AAATGCTGGACAGCGTGCCGCAGAAGCGTT…",144,110,27,0,143,134,71,262,267,0.144003,0.053636,0.167273,0.0,0.245455,1.0,0.677394,0.175513,false,false,false,false,false,false,false,true,null,null,"""Artifact suspicious""",null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ORD-2131606-01""","""chr17""",29483000,"""G""","""T""","""NF1""",false,2006,"""61-1G>T""","""splice site 61-1G>T""",0.0957,"""splice""","""NM_001042492""","""+""",false,"""1-snv""","""N""","""TTTTTTTTCTTTTTTTTTCATCTTCCAATA…",144,2299,464,0,143,143,71,599,596,0.070046,0.115789,0.038669,0.0,0.201827,1.0,1.0,1.0,false,false,false,false,false,false,false,true,null,null,"""Artifact suspicious""",null
"""ORD-2131606-01""","""chr2""",225422558,"""TTTCATCCATGGTCATC""","""T""","""CUL3""",false,1027,"""67-1_81delGATGACCATGGATGAA""","""splice site 67-1_81delGATGACCA…",0.0351,"""splice""","""NM_003590""","""-""",false,"""16-del""","""N""","""CCAAATGCTGTTTACATATTTTGTAATATC…",144,185,52,0,125,126,80,277,141,0.069294,0.115676,0.046486,0.0,0.281081,2.0150e-34,6.6014e-10,0.000001,true,false,false,false,false,false,false,true,"""Artifact suspicious""",""

257

In [26]:
# Filter 1234
arti_filter_1234 = final_df.filter(~pl.col("msec_filter_1234").is_null())
display(arti_filter_1234)
# Number of Samples with >= 1 artifacts
arti_filter_1234["sample"].n_unique()

sample,chrom,pos,ref,alt,gene,is_vus,depth,cds_effect,protein_effect,allele_fraction,functional_effect,transcript,strand,equivocal,mut_type,simplerepeat_trf,neighborhood_sequence,read_length,total_read,soft_clipped_read,flag_hairpin,pre_support_length,post_support_length,short_support_length,pre_farthest,post_farthest,low_quality_base_rate_under_q18,low_quality_pre,low_quality_post,distant_homology_rate,soft_clipped_rate,prob_filter_1,prob_filter_3_pre,prob_filter_3_post,filter_1_mutation_intra_hairpin_loop,filter_2_hairpin_structure,filter_3_microhomology_induced_mutation,filter_4_highly_homologous_region,filter_5_soft_clipped_reads,filter_6_simple_repeat,filter_7_mutation_at_homopolymer,filter_8_low_quality,msec_filter_123,msec_filter_1234,msec_filter_all,comment
str,str,i64,str,str,str,bool,i64,str,str,f64,str,str,str,bool,str,str,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,bool,bool,bool,bool,bool,bool,bool,bool,str,str,str,str
"""ORD-1888387-01""","""chr17""",7578545,"""CAGGGGAGTA""","""C""","""TP53""",false,6789,"""376_384delTACTCCCCT""","""Y126_P128del""",0.001,"""nonframeshift""","""NM_000546""","""-""",false,"""9-del""","""N""","""GCAAAACATCTTGTTGAGGGCCTGTAGGAA…",144,62,4,0,128,110,60,128,110,0.023858,0.030645,0.019355,0.0,0.064516,1.9162e-12,3.8711e-7,1.6516e-7,true,false,false,false,false,false,false,false,"""Artifact suspicious""","""Artifact suspicious""","""Artifact suspicious""",""" filter 3: p is small, but sup…"
"""ORD-1889526-01""","""chr22""",29090061,"""G""","""A""","""CHEK2""",false,4800,"""1420C>T""","""R474C""",0.0015,"""missense""","""NM_007194""","""-""",false,"""1-snv""","""N""","""GGCTTCTTCTGTCGTAAAACATGCCTTTGG…",144,67,12,0,140,143,66,274,161,0.035448,0.038806,0.038806,0.58209,0.179104,0.017276,0.352409,0.251762,false,false,false,true,false,false,false,false,null,"""Artifact suspicious""","""Artifact suspicious""",null
"""ORD-1896953-01""","""chr7""",55242464,"""AGGAATTAAGAGAAGC""","""A""","""EGFR""",false,4092,"""2235_2249delGGAATTAAGAGAAGC""","""E746_A750del""",0.0015,"""nonframeshift""","""NM_005228""","""+""",false,"""15-del""","""N""","""TAAAATTCCCGTCGCTATCAAAACATCTCC…",144,35,14,0,97,133,69,330,513,0.021429,0.025714,0.017143,0.0,0.4,0.000034,0.000006,4.7580e-8,false,false,true,false,false,false,false,false,"""Artifact suspicious""","""Artifact suspicious""","""Artifact suspicious""",null
"""ORD-1898635-01""","""chr16""",79633724,"""C""","""G""","""MAF""",true,819,"""76G>C""","""D26H""",0.0037,"""missense""","""NM_005360""","""-""",false,"""1-snv""","""N""","""CACTTCAAACTTCATCAGATGGAAGTCATT…",144,29,16,0,120,116,65,297,339,0.011734,0.013793,0.017241,0.0,0.551724,3.7601e-7,2.4400e-7,9.4594e-7,true,false,true,false,true,false,false,false,"""Artifact suspicious""","""Artifact suspicious""","""Artifact suspicious""",null
"""ORD-1899997-01""","""chr7""",6026959,"""G""","""C""","""PMS2""",true,2125,"""1437C>G""","""H479Q""",0.5002,"""missense""","""NM_000535""","""-""",false,"""1-snv""","""N""","""TCCGTAGGGTCACTGGGTCCCTGACTGGAA…",144,8033,380,0,143,143,71,438,515,0.016786,0.019109,0.015063,0.547492,0.047305,1.0,1.0,1.0,false,false,false,true,false,false,false,false,null,"""Artifact suspicious""","""Artifact suspicious""",null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ORD-2117808-01""","""chr3""",178952085,"""A""","""G""","""PIK3CA""",false,703,"""3140A>G""","""H1047R""",0.01,"""missense""","""NM_006218""","""+""",false,"""1-snv""","""N""","""GAAACAAATGAATGATGCACGTCATGGTGG…",101,29,0,0,60,90,49,219,326,0.017071,0.013793,0.027586,0.0,0.0,0.001003,3.1723e-9,3.0091e-9,false,false,true,false,false,false,false,false,"""Artifact suspicious""","""Artifact suspicious""","""Artifact suspicious""",null
"""ORD-2118842-01""","""chr2""",25463568,"""A""","""G""","""DNMT3A""",true,2024,"""2114T>C""","""I705T""",0.004,"""missense""","""NM_022552""","""-""",false,"""1-snv""","""N""","""CATTGCAGGGACTGCCCCCAGTCACCAGAT…",144,56,2,0,95,117,

77

In [27]:
summarize_res(final_df)

filter,percentage,n_failed,total_varints
str,f64,i64,i32
"""%_filter_1_mutation_intra_hair…",0.556019,40,7194
"""%_filter_2_hairpin_structure""",0.0,0,7194
"""%_filter_3_microhomology_induc…",0.639422,46,7194
"""%_filter_4_highly_homologous_r…",0.319711,23,7194
"""%_filter_5_soft_clipped_reads""",0.903531,65,7194
…,…,…,…
"""%_filter_7_mutation_at_homopol…",0.0139,1,7194
"""%_filter_8_low_quality""",2.34918,169,7194
"""%_msec_filter_123""",1.014735,73,7194
